In [35]:
import re
import json
from collections import defaultdict, Counter

class MalagasyNextWordPredictor:
    def __init__(self):
        # Table de transition : { "mot_actuel": { "mot_suivant": fréquence } }
        self.model = defaultdict(Counter)

    def train_on_text(self, text):
        """
        Analyse le texte pour construire les bi-grammes.
        """
        # Nettoyage et tokenisation (minuscules, garde l'apostrophe malgache)
        words = re.findall(r"\b[a-zA-ZàâéèêëîïôùûüÀÂÉÈÊËÎÏÔÙÛÜ’']+\b", text.lower())
        
        # Parcourt les mots pour créer les paires
        for i in range(len(words) - 1):
            current_word = words[i]
            next_word = words[i+1]
            self.model[current_word][next_word] += 1
            
        print(f"✅ Entraînement terminé : {len(words)} mots analysés.")

    def predict(self, last_word, limit=5):
        """
        Retourne les mots les plus probables après 'last_word'.
        """
        last_word = last_word.lower().strip()
        
        if last_word not in self.model:
            return []

        # Récupère les N mots les plus fréquents
        predictions = self.model[last_word].most_common(limit)
        return [word for word, count in predictions]

# --- 1. PRÉPARATION DES DONNÉES D'EXEMPLE ---
# En attendant ton dictionnaire de texte, voici un petit corpus pour tester
corpus_test = """
Isika dia mandeha any an-tsena. 
Isika dia faly miara-miasa. 
Isika dia tia mianatra zavatra vaovao.
Ny mpianatra dia mandeha mianatra.
Ny olona dia tia fety.
Mandeha any Antananarivo izahay rahampitso.
"""

# --- 2. INITIALISATION ET ENTRAÎNEMENT ---
predictor = MalagasyNextWordPredictor()
predictor.train_on_text(corpus_test)

# --- 3. TEST DE PRÉDICTION ---
def test_prediction(word):
    suggestions = predictor.predict(word)
    print(f"Après '{word}', suggestions : {suggestions}")

# Essaye ces mots
test_prediction("isika")
test_prediction("dia")
test_prediction("mandeha")

✅ Entraînement terminé : 32 mots analysés.
Après 'isika', suggestions : ['dia']
Après 'dia', suggestions : ['mandeha', 'tia', 'faly']
Après 'mandeha', suggestions : ['any', 'mianatra']


In [36]:
# --- 4. EXEMPLE AVEC CHARGEMENT DE FICHIER (À décommenter plus tard) ---
def charger_et_entrainer(chemin_fichier):
    with open(chemin_fichier, 'r', encoding='utf-8') as f:
        contenu = f.read()
        predictor.train_on_text(contenu)

In [37]:
charger_et_entrainer("corpus.txt")

✅ Entraînement terminé : 3036 mots analysés.


In [38]:
import json

def sauvegarder_modele(predictor, filename="model_ngram.json"):
    # On convertit le defaultdict/Counter en dictionnaire standard pour le JSON
    data_to_save = {word: dict(counts) for word, counts in predictor.model.items()}
    
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data_to_save, f, ensure_ascii=False, indent=4)
    print(f"💾 Modèle sauvegardé avec succès dans {filename}")

# Utilisation :
sauvegarder_modele(predictor)

💾 Modèle sauvegardé avec succès dans model_ngram.json
